# Exercise: AIMNet2 model internals

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">Learning objectives</div>
  <div style="padding:14px;"><p>Inspect the electrostatic head and compare dipoles from AIMNet2 charges.</p><p>Connect dipole and polarizability derivatives to infrared and Raman spectra.</p><p>Compare charge-response models and check how normalization changes their interpretation.</p></div>
</div>

**Model choice.** [AIMNet2](https://github.com/isayevlab/aimnetcentral) exposes atomic charges and electrostatic contributions, so we can inspect the quantities behind its predictions.

Complete the `...` placeholders in **Your turn** cells. Other cells are provided.

**Kernel:** AIMNet2. Run from this notebook's directory in a fresh kernel, using the workshop environment with the Psience response APIs installed.

We will compare charge-derived dipoles for water, then calculate a Raman spectrum for benzaldehyde using the same AIMNet2 calculator.

**Predict:** which molecular derivative determines infrared intensity? Which determines Raman intensity?

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.optimization import optimize_ase
_, OUTPUT = start_exercise()

import copy
import numpy as np
import torch
from ase.build import molecule
from ase.optimize import BFGS
from aimnet.calculators import AIMNet2ASE, AIMNet2Calculator

# Set precision before loading the model for the response calculation.
torch.set_default_dtype(torch.float64)
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '4')))

## 1. Inspect the electrostatic head

The default calculator keeps its long-range Coulomb term outside the torch model. The model also contains a short-range correction. Inspect both before exposing the Coulomb energy as a separate output.

In [ ]:
calculator = AIMNet2Calculator('aimnet2', device='cpu')
model = calculator.model
print(model)

Copy the coupling module into `model.outputs`, assign a new output key, and register it with the calculator. This keeps the original total-energy calculation in place.

In [ ]:
print('Model output modules:', list(model.outputs))
coupling_head = calculator.external_coulomb
print('Electrostatic coupling module:', coupling_head)
print('Charge input key:', coupling_head.key_in)
print('Short range correction:', model.outputs['srcoulomb'])

probe = copy.deepcopy(coupling_head)
probe.key_out = 'electrostatic_energy'
model.outputs['electrostatic_probe'] = probe
calculator.keys_out = [*calculator.keys_out, 'electrostatic_energy']

<div role="alert" class="alert alert-warning">AIMNet2 model families can organize charge and Coulomb modules differently. Reusing this exact head path with another model requires inspecting that model first.</div>

<div role="alert" class="alert alert-info">The copied Coulomb module writes a separate electrostatic energy. The original Coulomb and total-energy paths remain active.</div>

## 2. Optimize water and compare dipoles

The electrostatic module uses AIMNet2's `charges` tensor. Its dipole is $\sum_i q_i \mathbf{r}_i$. ASE calculates a dipole from the same charges and coordinate origin. Predict whether the two vectors will agree for neutral water.

Start with the same ASE workflow as the other exercises: build a structure, attach the calculator, and relax it.

In [ ]:
water = molecule('H2O')
water.calc = AIMNet2ASE(base_calc=calculator, charge=0)
optimize_ase(water, optimizer=BFGS, fmax=0.00001)

From here we can compute partial charges and track the electrostatic contributions

**Your turn.** Retrieve partial charges and the dipole from the ASE water object. Tensor handling, comparisons, and visualization are supplied.


In [ ]:
inputs = {'coord': water.positions, 'numbers': water.numbers, 'charge': 0.0}
result = calculator.eval(inputs)

electrostatic_charges = result[coupling_head.key_in].detach().cpu().numpy()
electrostatic_dipole = np.einsum('i,ij->j', electrostatic_charges, water.positions)

partial_charges = ...  # TODO: retrieve atomic charges
partial_charge_dipole = ...  # TODO: retrieve the dipole

print('Total energy (eV):', result['energy'].item())
print('Electrostatic coupling energy (eV):', result['electrostatic_energy'].item())
print('Electrostatic-head charges (e):', electrostatic_charges)
print('AIMNet2 partial charges (e):', partial_charges)
print('Electrostatic-charge dipole (e·Å):', electrostatic_dipole)
print('AIMNet2 partial-charge dipole (e·Å):', partial_charge_dipole)
print('Difference (e·Å):', electrostatic_dipole - partial_charge_dipole)
np.testing.assert_allclose(electrostatic_charges, partial_charges, rtol=1e-6, atol=1e-6)
np.testing.assert_allclose(electrostatic_dipole, partial_charge_dipole, rtol=1e-6, atol=1e-6)

<div role="alert" class="alert alert-success">The charge and dipole comparisons passed: both calculations use the same AIMNet2 partial charges for neutral water.</div>

## 3. Plot the dipole and infrared spectrum

Psience's `Molecule` stores coordinates in Bohr, while ASE positions are in Å. Convert the geometry, then use the native X3D view. The two dipoles coincide, so we display one vector scaled by five.

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #ffeeba; background-color:#fff3cd; color:#856404;">Reading the dipole view</div>
  <div style="padding:14px;">The plotted dipole is multiplied by five so its direction is visible beside the molecule. The numeric dipole printed above remains in e&#183;&#197; and is not rescaled.</div>
</div>

In [ ]:
from McUtils.Data import UnitsData
from Psience.Molecools import Molecule

water_molecule = Molecule(
    water.get_chemical_symbols(),
    water.positions * UnitsData.convert('Angstroms', 'BohrRadius'),
    bonds=[(0, 1), (0, 2)],
    energy_evaluator='aimnet2',
    dipole_evaluator='aimnet2'
)
water_molecule.plot(backend='x3d', dipole=5*partial_charge_dipole)

Infrared intensity depends on dipole derivatives along the normal modes. Psience calculates the frequencies and harmonic spectrum through the molecular interface.

In [ ]:
water_molecule.get_normal_modes().freqs * UnitsData.convert("Hartrees", "Wavenumbers")

In [ ]:
water_molecule.get_harmonic_spectrum().plot()

## 4. Load and relax benzaldehyde

The supplied XYZ is a planar optimization warm start. We will refine it, compute the Hessian, and project out translation and rotation to obtain 36 vibrational modes.

**Your turn.** Convert the ASE positions from Å to Bohr when creating the Psience molecule, as in the water example.

In [ ]:
from ase.io import read
from Psience.ElectronicStructure import AIMNet2Backend
from Psience.Molecools.Evaluator import AIMNet2EnergyEvaluator

A2B = UnitsData.convert('Angstroms', 'BohrRadius')
H2W = UnitsData.convert('Hartrees', 'Wavenumbers')
benzaldehyde_atoms = read('benzaldehyde.xyz')
mol = Molecule(benzaldehyde_atoms.get_chemical_symbols(),
    benzaldehyde_atoms.positions * ...,  # TODO: convert Å to Bohr
    charge=0, spin=1, guess_bonds=True)
backend = AIMNet2Backend(mol.atoms, model=calculator, charge=0,
    multiplicity=1, polarizability_mode='total_energy')
energy = AIMNet2EnergyEvaluator.from_mol(mol, backend=backend)

In [ ]:
mol = mol.optimize(evaluator=energy, method='quasi-newton',
    line_search=False, tol=2e-6, max_iterations=160,
    max_displacement=.03, reembed=False)
E, G, H = mol.calculate_energy(evaluator=energy, order=2)
print('Energy (Ha):', E)
print('Maximum absolute gradient (Ha/Bohr):', np.max(np.abs(G)))
assert np.max(np.abs(G)) < 2e-6, 'Check geometry convergence.'
mol = Molecule(mol.atoms, mol.coords, charge=0, spin=1,
    energy=E, potential_derivatives=[G, H], guess_bonds=True)
modes = mol.get_normal_modes(use_internals=False, project_transrot=True)
print('Frequencies (cm^-1):', modes.freqs * H2W)
assert len(modes.freqs) == 36 and np.all(modes.freqs > 0)
mol.plot(backend='x3d')

## 5. Compare charge-response models

Polarizability describes the dipole's response to an electric field. We will compare three choices of charge-dependent energy at the same geometry:

| Mode | Energy used for charge response |
|:--|:--|
| `coulomb` | External Coulomb contribution |
| `net_coulomb` | Coulomb contribution plus its short-range correction |
| `total_energy` | Conditional total energy after changing charges before the final neural pass |

<div role="alert" class="alert alert-warning">This charge-response construction is experimental. Native AIMNet2 charges need not minimize the conditional energy. The Coulomb controls can have unstable curvature, so we explicitly enable them for diagnosis.</div>

**Your turn.** Pass the current loop's response-mode name to the polarizability evaluator. The derivative order and diagnostic calls are supplied.

In [ ]:
evaluator = mol.get_polarizability_evaluator('aimnet2', backend=backend)
coords = np.asarray(mol.coords) / A2B  # evaluator input is in Å
responses, diagnostics = {}, {}
for mode in ['coulomb', 'net_coulomb', 'total_energy']:
    mu, alpha = evaluator.evaluate(coords, order=1,
        polarizability_mode=..., allow_indefinite=True)
    responses[mode] = alpha  # alpha and d(alpha)/dR in atomic units
    _, diagnostics[mode] = backend.evaluate_polarizability(coords,
        polarizability_mode=mode, allow_indefinite=True,
        return_diagnostics=True)

In [ ]:
from McUtils.Formatters import TableFormatter
rows = []
for mode, d in diagnostics.items():
    curvature = d['constrained_eigenvalues']
    rows.append([mode, bool(d['stable']), int(np.sum(curvature < 0)),
        float(min(curvature)),
        float(np.max(np.abs(d['linearized_zero_field_charge_shift'])))])
print(TableFormatter(['', '', '{:d}', '{:.4g}', '{:.4g}']).format(
    ['mode', 'stable', 'negative', 'min curvature (eV/e²)', 'max linearized shift (e)'], rows))
print('Native dipole (e Bohr):', mu[0])
print('Total-energy polarizability (atomic units):')
print(responses['total_energy'][0])

Negative eigenvalues identify unstable charge directions that conserve total charge. Check both the stability label and the linearized zero-field charge shift before interpreting a response.

**Discuss:** which models have negative directions? Why does this planar point-charge model have zero perpendicular static polarizability? Does positive curvature by itself establish physical accuracy?

## 6. Calculate harmonic Raman spectra

Raman intensity depends on polarizability derivatives along the normal modes. Use a 532 nm pump and 298 K temperature for all three response models.

**Your turn.** Supply each mode's polarizability expansion from `responses` to the Raman calculation. Frequencies and pump frequency enter this API in Hartrees.

In [ ]:
LASER_NM = 532.
TEMPERATURE = 298.
spectra = {}
for mode in responses:
    spectra[mode] = mol.get_harmonic_raman_spectrum(modes=modes,
        polarizability_derivatives=...,  # TODO: this mode's expansion
        pump_frequency=1e7 / LASER_NM / H2W,
        temperature=TEMPERATURE)

Normalize each line spectrum to its strongest line, then apply Gaussian broadening with `breadth=10` cm⁻¹. This makes the shapes easier to compare. Keep the original intensities for the scale comparison below.

<div role="alert" class="alert alert-warning">Normalization hides intensity scale. Curves labeled unstable are diagnostic controls. The total-energy response also remains conditional and experimental.</div>

In [ ]:
plot = None
for mode, spectrum in spectra.items():
    label = mode + ('' if diagnostics[mode]['stable'] else ' (unstable)')
    plot = spectrum.normalize().broaden(
        breadth=10, broadening_type='gaussian').plot(
        figure=plot, freq_min=0, freq_max=3400,
        plot_style={'label': label}, plot_legend=True,
        axes_labels=['Raman shift (cm$^{-1}$)', 'Intensity / strongest line'])
plot

In [ ]:
reference_sum = spectra['total_energy'].intensities.sum()
print(TableFormatter(['', '{:.6g}']).format(
    ['mode', 'sum / total-energy sum'],
    [[mode, float(s.intensities.sum() / reference_sum)]
     for mode, s in spectra.items()]))

for mode, spectrum in spectra.items():
    np.savetxt(OUTPUT / f'{mode}_raman.csv',
        np.c_[spectrum.frequencies, spectrum.intensities], delimiter=',',
        header='frequency_cm1,relative_stokes_intensity', comments='')
    np.savez(OUTPUT / f'{mode}_response.npz',
        alpha=responses[mode][0], dalpha=responses[mode][1])
np.savez(OUTPUT / 'aimnet_nuclear.npz', atoms=mol.atoms,
    coords_bohr=np.asarray(mol.coords), energy_hartree=E,
    gradient=G, hessian=H, frequencies_cm1=modes.freqs * H2W)
plot.savefig(str(OUTPUT / 'raman_comparison.png'))

## Try, explain, and report

- Compare the normalized curves with the intensity sums. Which response has the largest intensity scale?
- Change `LASER_NM` or `TEMPERATURE`, rerun the Raman cells, and explain what changes.
- Why do the three models share frequencies but predict different intensities?
- What additional evidence would you want before comparing these spectra with experiment?

# Summary

<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #c3e6cb; background-color:#d4edda; color:#155724;">What you learned</div>
  <div style="padding:14px;"><p>You exposed the electrostatic contribution and compared dipoles from the same AIMNet2 charges.</p><p>You calculated infrared and Raman spectra through Psience's molecular APIs.</p><p>You checked charge-response stability and compared intensity scales alongside normalized spectra.</p></div>
</div>